# Ollama on Google Colab with Persistent Models in Google Drive

This notebook installs and runs [Ollama](https://ollama.com) in Google Colab **and** stores models in Google Drive so you don’t need to re-download them every session.

### Requirements
- Recommended: **Colab Pro/Pro+** with GPU runtime (T4 / L4 / A100 etc.).
- Enough Google Drive space for your models.

---
## 0. (Optional) Check GPU
You can run this to verify a GPU is available.


In [ ]:
# Check if a GPU is available
!nvidia-smi || echo 'No GPU available. You may be on a CPU-only runtime.'

Mon Dec  8 11:29:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Mount Google Drive
This connects your Google Drive so we can store Ollama models there persistently.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Install Ollama
Ollama must be installed on each new Colab runtime (the binary itself cannot be made persistent), but the **models** will be saved in Drive.


In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## 3. Point Ollama's model directory to Google Drive
By default, Ollama stores models under `~/.ollama/models` inside the ephemeral VM.
We:
1. Create a folder in Google Drive (e.g. `MyDrive/ollama_models`).
2. Remove the default `~/.ollama/models` directory.
3. Create a **symlink** so `~/.ollama/models` points into Google Drive.

Run this **once per session** before pulling models.


In [ ]:
# Create a persistent models directory in Google Drive
!mkdir -p /content/drive/MyDrive/RAG/ollama_models


# Remove any existing ephemeral models directory
!rm -rf ~/.ollama/models

# Symlink Ollama's model directory to Google Drive
!ln -s /content/drive/MyDrive/RAG/ollama_models Notebooks/RAG/ollama_models ~/.ollama/models

print('Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models')

ln: target '/root/.ollama/models': No such file or directory
Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models


## 4. Start the Ollama server
We start `ollama serve` as a background process so we can interact with it from the notebook.


In [ ]:
import subprocess, time

server = subprocess.Popen(['ollama', 'serve'])
print('Ollama server started as background process (PID:', server.pid, ')')
time.sleep(5)
print('Waited 5 seconds for the server to initialize.')

Ollama server started as background process (PID: 2463 )
Waited 5 seconds for the server to initialize.


## 5. Pull a model (stored in Google Drive)
Now when you pull a model, its files will be stored in `MyDrive/ollama_models` and **persist across sessions**.
Here we use `llama3.1` as an example. You can swap it for any other model name available in Ollama.


In [ ]:
# Download a model (this will be stored in Google Drive)
!ollama pull gemma3:4b
!ollama pull deepseek-r1:1.5b
!ollama pull gemma3:12b
!ollama pull llama3.2

In [ ]:
!ollama pull deepseek-r1:1.5b

In [ ]:
!ollama pull gemma3:12b

In [ ]:
!ollama pull llama3.2

## 6. Quick test prompt via CLI
Confirm the model works by running a simple prompt.


In [ ]:
!ollama run deepseek-r1:1.5b "Give me three bullet points about retrieval-augmented generation."


Okay, here are three key bullet points about Retrieval-Augmented Generation (RAG):

*   **Combines Knowledge Retrieval with Generative AI:** RAG systems don’t rely solely on the knowledge encoded within their large language model (LLM). Instead, they first *retrieve* relevant information from an external knowledge source (like a database, website, or document collection) and then *use* that retrieved information to inform the LLM's response generation.

*   **Improves Accuracy and Reduces Hallucinations:** By grounding the LLM’s output in factual data, RAG significantly reduces the risk of the model “hallucinating” or inventing information. It ensures the generated text is based on verifiable sources.

*   **Enhances Context and Specificity:** RAG allows for tailoring responses to very specific contexts.  The retrieved information provides a richer context for the LLM, leading to more relevant, detailed, and useful answers compared to a purely generative model.


Would you like me to 

## 7. Python helper for calling Ollama
You can also call the model from Python using `subprocess`. This keeps things simple while still letting you integrate Ollama into other pipelines.


In [ ]:
import subprocess

def ollama_chat(model: str, prompt: str) -> str:
    """Send a single prompt to an Ollama model and return the response as text."""
    result = subprocess.run(
        ['ollama', 'run', model],
        input=prompt,
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Ollama error: {result.stderr}")
    return result.stdout

response = ollama_chat('deepseek-r1:1.5b', 'List three use cases of running Ollama models in Google Colab.')
print(response)

Thinking...
Okay, so I need to come up with three use cases for using Ollama models in Google Colab. Hmm, let me think about this.

First, I know that Ollama is a faster alternative to Python's scikit-learn and other machine learning libraries. It's built on top of PyTorch, which means it should be more efficient. So maybe one use case would involve using Ollama for something really intensive or where speed matters a lot.

I remember that in some cases, you might want to run models on a cloud server instead of your local machine, especially if the data is big or the computation needs to be done offsite. Maybe one idea is running an Ollama model locally but using a cloud resource for the data. That way, it's manageable on your computer while still leveraging Ollama's speed.

Another thought: sometimes people might use multiple models in sequence or work with different datasets. So maybe another use case involves comparing multiple Ollama models across different datasets to see which per

## 8. (Optional) Minimal RAG-style example
A tiny demonstration of retrieval-augmented generation:
1. Embed a small in-memory corpus with `sentence-transformers`.
2. Retrieve the most relevant chunks.
3. Pass the context + question to the Ollama model.
You can expand this later with PDFs, vector databases, etc.


In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import textwrap
import subprocess

documents = [
    "Retrieval-augmented generation (RAG) combines a retriever and a generator.",
    "Ollama can run local language models and expose them via a simple CLI.",
    "Vector databases like FAISS or Chroma are often used for efficient similarity search.",
]

embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedder.encode(documents, convert_to_numpy=True)

def retrieve_context(query: str, k: int = 2):
    q_emb = embedder.encode([query], convert_to_numpy=True)[0]
    sims = doc_embeddings @ q_emb / (
        np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(q_emb) + 1e-10
    )
    top_idx = np.argsort(-sims)[:k]
    return [documents[i] for i in top_idx]

def ollama_rag(model: str, question: str) -> str:
    context_chunks = retrieve_context(question, k=2)
    context = "\n".join(f"- {c}" for c in context_chunks)
    prompt = textwrap.dedent(f"""
    You are an assistant with access to the following context:
    {context}

    Using ONLY this context when possible, answer the question:
    {question}
    """)
    result = subprocess.run(
        ['ollama', 'run', model],
        input=prompt,
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Ollama error: {result.stderr}")
    return result.stdout

answer = ollama_rag('llama3.1', 'How does RAG combine retrieval and generation?')
print(answer)

## 9. (Optional) Stop the Ollama server
If you want to stop the Ollama server without restarting the whole runtime, you can terminate the process.


In [ ]:
try:
    server.terminate()
    print('Terminated Ollama server process.')
except NameError:
    print('No server process variable found. It may already be stopped or the runtime was restarted.')

Terminated Ollama server process.
